### Train Your Own AI Model with the CSU-IR Pre-Training Notebook!

💡 **Pro Tip:** For this training task, using a GPU isn't just a tip—it's **highly recommended** for a fast and efficient experience!

Just click on **Runtime** at the top-right of the page, select **Change runtime type**, and choose **T4 GPU** from the *Hardware accelerator* dropdown. This will supercharge your training process.

<hr style="border: none; border-top: 2px dashed #ccc;">

#### 🚀 Training Launch Sequence (Crucial Steps!)

1.  **Activate Your GPU Runtime**: In the top menu, go to **Runtime** → **Change runtime type** and select **T4 GPU**. After this, ensure you're connected. Colab often connects automatically, but if you don't see `Connecting...` at the top-right, please click the **Connect** button manually.

2.  **Wait for the Green Checkmark** (✓): Wait a moment until a green checkmark appears next to your `RAM` and `Disk` status. This confirms your powerful GPU-enabled environment is **"Connected"** and ready.

3.  **Run the Setup Cell**: Execute the first code cell, typically labeled **Part I: Environment Setup & Data Download**. This will install all necessary tools and fetch the datasets.

4.  **Choose Your Data & Start Training**: Once the setup is complete, proceed to the next code cells where you can select the dataset you want to use and then run the training cell to begin the process.

<hr style="border: none; border-top: 2px dashed #ccc;">

#### ✨ After starting the training, get ready to see:

Our intelligent pipeline will begin the pre-training process, and you'll be able to monitor its progress in real-time:

*   **1. Live Progress Updates**: Watch the model learn epoch by epoch, with continuous feedback on its training loss and other key metrics.
*   **2. Model Checkpoints Saved**: As the model trains, checkpoints (snapshots of the model's "brain") will be periodically saved.
*   **3. Your Custom Model**: At the end of the process, you will have your very own pre-trained CSU-IR model weights, saved and ready for use.

<div style="background-color: #fffbe6; border-left: 6px solid #ffc107; padding: 15px; margin: 20px 0; border-radius: 5px;">
    <h4 style="color: #856404; margin-top: 0;">
        📝 A Note on Dataset Size & Full Training
    </h4>
    <p>
        To prevent exceeding Google Drive's storage limits, the first stage of MD data pre-training in this notebook uses a sample of the data for demonstration (specifically, the first 1/5 of each dataset).
    </p>
    <p>
        For instructions on performing a complete training run with the full dataset, please follow these steps:
    </p>
    <ol>
        <li><strong>Download the complete dataset</strong> from our Hugging Face Repository: <a href="https://huggingface.co/Skylight666/CSU-IR/tree/main" target="_blank">CSU-IR on Hugging Face</a>.</li>
        <li>Place the downloaded files into their corresponding data folders.</li>
        <li><strong>Follow the detailed instructions</strong> in the <code>train</code> section of our <code>README</code> file, under the <code>Local Installation and Usage</code> guide.</li>
    </ol>
</div>

---

### **Step 1: Project Setup & Data Download** ⚙️

<div style="background-color: #e6f7ff; border-left: 6px solid #1890ff; padding: 15px; margin: 15px 0; border-radius: 5px;">
    <h4 style="color: #0050b3; margin-top: 0; font-size: 1.1em;">
    </h4>
    <p>
        In this step, we'll supercharge your Colab environment for the CSU-IR project. We're about to do two key things to get you ready for training:
    </p>
    <ul style="margin-top: 0; padding-left: 20px;">
        <li><strong>🚀 Install the Essentials</strong>: We'll quickly add the special libraries this project needs to run at full speed.</li>
        <li><strong>🧠 Assemble the Core Engine</strong>: We'll download the pre-trained model weights from Hugging Face, which are the core intelligence of our tool.</li>
    </ul>
    <p style="font-weight: bold;">
        Just run the code cell below, and we'll handle the rest.
    </p>
</div>

In [ ]:
#@title 🧠 1: Setting Up Your Workspace
# ==============================================================================
#  CSU-IR Project Setup Script for Colab
# ==============================================================================
import os
import shutil
from google.colab import drive
from google.colab import userdata
from huggingface_hub import hf_hub_download # Import at the top

# ==================================================================
# --- 1. Mount Google Drive ---
# ==================================================================
print("📁 Mounting Google Drive...")
drive.mount('/content/drive', force_remount=True) # Use force_remount for robustness
print("✅ Google Drive mounted.")


# ==================================================================
# --- 2. Define All Project Paths ---
# ==================================================================
print("\n--- Defining Project Paths ---")
# Project Root
GDRIVE_REPO_PATH = "/content/drive/MyDrive/colab_repos/CSU-IR"

# Requirements file
REQUIREMENTS_FILE = os.path.join(GDRIVE_REPO_PATH, 'requirements_colab.txt')

# --- Data and Model Destination Folders on Google Drive ---
# These are the folders where downloaded files will be saved.
DEST_PRETRAIN_MD_PATH = os.path.join(GDRIVE_REPO_PATH, 'CSU-IR', 'data', 'pretrain_data', 'colab_MD_data')
DEST_PRETRAIN_DFT_PATH = os.path.join(GDRIVE_REPO_PATH, 'CSU-IR', 'data', 'pretrain_data', 'Density functional simulation data')
print("✅ Paths defined.")


# ==================================================================
# --- 3. Clone or Update the 'CSU-IR' repository ---
# ==================================================================
print("\n--- Cloning or Updating Repository ---")
try:
    GIT_TOKEN = userdata.get('GITHUB_PAT')
except userdata.SecretNotFoundError:
    raise Exception("🛑 Error: Please set your GitHub Personal Access Token in Colab Secrets (secret name: GITHUB_PAT)")

if not os.path.exists(GDRIVE_REPO_PATH):
    print(f"⏳ Cloning 'CSU-IR' to: {GDRIVE_REPO_PATH}")
    !git clone -q https://{GIT_TOKEN}@github.com/Hsqcsu/CSU-IR.git {GDRIVE_REPO_PATH}
else:
    print(f"✅ Repository already exists. Pulling latest changes...")
    !cd {GDRIVE_REPO_PATH} && git pull


# ==================================================================
# --- 4. Install Dependencies ---
# ==================================================================
print("\n--- Installing Dependencies ---")
if os.path.exists(REQUIREMENTS_FILE):
    print("⏳ Installing dependencies from requirements_colab.txt...")
    !pip install -q -r {REQUIREMENTS_FILE}
    print("✅ Dependencies installed.")
else:
    print(f"⚠️ Warning: Could not find requirements file at {REQUIREMENTS_FILE}.")


# ==================================================================
# --- 5. Define Download Helper and File Lists ---
# ==================================================================
print("\n--- Preparing for Download ---")
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except userdata.SecretNotFoundError:
    raise Exception("🛑 Error: Please set your Hugging Face Token in Colab Secrets (secret name: HF_TOKEN)")

# --- Reusable Download Helper Function ---
def download_files_from_hf(repo_id, files_dict, destination_dir, token):
    """
    Downloads a dictionary of files from a Hugging Face repo to a destination directory.
    """
    print(f"\n--- Checking files for: {os.path.basename(destination_dir)} ---")
    os.makedirs(destination_dir, exist_ok=True)

    for hf_path, local_name in files_dict.items():
        target_path = os.path.join(destination_dir, local_name)
        if not os.path.exists(target_path):
            print(f"  -> Downloading '{local_name}'...")
            try:
                downloaded_path = hf_hub_download(repo_id=repo_id, filename=hf_path, token=token)
                shutil.copy(downloaded_path, target_path)
                print(f"  ✅ Downloaded successfully.")
            except Exception as e:
                print(f"  ❌ FAILED to download '{local_name}'. Error: {e}")
        else:
            print(f"  👍 '{local_name}' already exists.")

# --- Define All Files to be Downloaded ---
HF_REPO_ID = "Skylight666/CSU-IR"
PRETRAIN_DATA_MD_TO_DOWNLOAD = {
    "Pretrain_data/colab_MD_data/USTPO_train_ir.pt": "USTPO_train_ir.pt",
    "Pretrain_data/colab_MD_data/USTPO_train_smiles.txt": "USTPO_train_smiles.txt",
    "Pretrain_data/colab_MD_data/USTPO_val_ir.pt": "USTPO_val_ir.pt",
    "Pretrain_data/colab_MD_data/USTPO_val_smiles.txt": "USTPO_val_smiles.txt",
    "Pretrain_data/colab_MD_data/USTPO_test_ir.pt": "USTPO_test_ir.pt",
    "Pretrain_data/colab_MD_data/USTPO_test_smiles.txt": "USTPO_test_smiles.txt"
}
PRETRAIN_DATA_DFT_TO_DOWNLOAD = {
    "Pretrain_data/Stage 2 - Density functional simulation data - QM9S/QM9S_DFT_train_ir.pt": "QM9S_DFT_train_ir.pt",
    "Pretrain_data/Stage 2 - Density functional simulation data - QM9S/QM9S_DFT_train_smiles.txt": "QM9S_DFT_train_smiles.txt",
    "Pretrain_data/Stage 2 - Density functional simulation data - QM9S/QM9S_DFT_val_ir.pt": "QM9S_DFT_val_ir.pt",
    "Pretrain_data/Stage 2 - Density functional simulation data - QM9S/QM9S_DFT_val_smiles.txt": "QM9S_DFT_val_smiles.txt",
    "Pretrain_data/Stage 2 - Density functional simulation data - QM9S/QM9S_DFT_test_ir.pt": "QM9S_DFT_test_ir.pt",
    "Pretrain_data/Stage 2 - Density functional simulation data - QM9S/QM9S_DFT_test_smiles.txt": "QM9S_DFT_test_smiles.txt"
}
print("✅ Download helper and file lists are ready.")

# ==================================================================
# --- 6. Execute All Downloads ---
# ==================================================================
print("\n--- Executing All Downloads ---")

download_files_from_hf(HF_REPO_ID, PRETRAIN_DATA_MD_TO_DOWNLOAD, DEST_PRETRAIN_MD_PATH, HF_TOKEN)
download_files_from_hf(HF_REPO_ID, PRETRAIN_DATA_DFT_TO_DOWNLOAD, DEST_PRETRAIN_DFT_PATH, HF_TOKEN)

# ==================================================================
# --- Finalization ---
# ==================================================================
print("\n\n🎉🎉🎉 Full project setup is complete! 🎉🎉🎉")

---
### **Step 2: Environment Setup & Model Initialization**
<div style="background-color: #e6f7ff; border-left: 6px solid #1890ff; padding: 15px; margin: 15px 0; border-radius: 5px;">
    <h4 style="color: #0050b3; margin-top: 0;">
        💡 A Note on Initialization
    </h4>
    <p>
        After you run this cell, you might see some error messages appear in the output. <strong>Please don't worry, this is expected.</strong>
    </p>
    <p>
        We have already handled this initialization process within the code. You don't need to do anything.
    </p>
    <p>
        <strong>Simply wait for this cell to finish running, and then proceed to the next cell.</strong>
    </p>
</div>

In [ ]:
# @title 🧠 2. Environment Setup & Model Initialization
# @markdown This cell is responsible for setting up all paths, importing custom modules, and initializing the models.

print("⏳ Initializing and warming up the RDKit environment...")
try:
    from rdkit import Chem
    # If the first import succeeds, great.
except ImportError as e:
    # The first import may fail in a freshly restarted Colab runtime.
    # This is a known issue. We capture the error and retry,
    # as the failed attempt often prepares the environment for the next one.
    print(f"Caught an expected initialization error: {e}")
    print("Retrying import...")
    from rdkit import Chem

print("✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅")
print("The code is running normally")
print("💡💡💡 This is a normal phenomenon, please don't be surprised, wait for the program to finish running and run the next cell.!")
print("-------------------------------------------------------------------------")
print("✅ RDKit environment is ready!")

import os
import sys
import json
import gc
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR, CosineAnnealingWarmRestarts
from torch.amp import autocast, GradScaler
from tqdm.notebook import tqdm # Use tqdm.notebook for better display in Colab

print("--- Step 1: Setting up paths and importing modules ---")

# --- Path Definitions ---
try:
    from google.colab import drive
    drive.mount('/content/drive')
    GDRIVE_REPO_PATH = "/content/drive/MyDrive/colab_repos"
    print("✅ Google Drive mounted successfully.")
except ImportError:
    GDRIVE_REPO_PATH = "path/to/your/local/parent/folder"
    print("⚠️ Not in a Colab environment. Please ensure GDRIVE_REPO_PATH is set correctly.")

# --- Module and Data Paths ---
PROJECT_ROOT = os.path.join(GDRIVE_REPO_PATH, 'CSU-IR', 'CSU-IR')
MODEL_MODULE_PATH = os.path.join(PROJECT_ROOT, 'model')
UTILS_MODULE_PATH = os.path.join(PROJECT_ROOT, 'train_and_val')
roberta_tokenizer_path = os.path.join(MODEL_MODULE_PATH, 'tokenizer-smiles-roberta-1e_new')
MODEL_OUTPUT_PATH = os.path.join(PROJECT_ROOT, 'check_points', 'colab_trained')
os.makedirs(MODEL_OUTPUT_PATH, exist_ok=True)

# --- Module Imports ---
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
try:
    from model.SMILES_encoder import SmilesModel as Smiles_Model
    from model.IR_encoder import IRModel as IR_Model
    from train_and_val.SmilesEnumerator import augment_smiles, SmilesEnumerator
    print("✅ Custom modules imported successfully!")
except (ModuleNotFoundError, ImportError) as e:
    print(f"❌ Module import failed! Error: {e}")
    raise e

print("\n--- Step 2: Initializing models and utilities on CPU ---")
# Instantiate models
ir_model = IR_Model(output_channels=768, channels=32, dim=1024, no_txtnorm=False)
smiles_model = Smiles_Model(
    roberta_model_path=None,
    roberta_tokenizer_path=roberta_tokenizer_path,
    smiles_maxlen=300,
    max_position_embeddings=505,
    vocab_size=181,
    feature_dim=768,
)
# Instantiate SMILES enumerator for data augmentation
sme = SmilesEnumerator()

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# @title 3. Training Components & Core Loop Definition
# @markdown This cell defines the optimizer, schedulers, and the core training and validation functions,
# @markdown maintaining the exact same logic as the local script for consistency.

print("--- Step 3: Defining training components and core loops ---")

# --- Training Components ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"  -> Target device: {device}")

ir_model.to(device)
smiles_model.to(device)
print(f"  -> Models moved to device.")

optimizer = AdamW(list(smiles_model.parameters()) + list(ir_model.parameters()), lr=5e-5, weight_decay=1e-4)
scaler = GradScaler()
def lr_lambda(epoch): return epoch / 10 if epoch < 10 else 1.0
scheduler_warmup = LambdaLR(optimizer, lr_lambda=lr_lambda)
scheduler_cosine = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)
print("✅ Optimizer, schedulers, and GradScaler defined.")


# --- Core Training and Validation Functions (Logic identical to the local script) ---

def validate_model(smiles_model, ir_model, val_loader, device):
    smiles_model.eval()
    ir_model.eval()
    running_loss = 0.0
    result_smiles_features = []
    result_ir_features = []
    # Using tqdm.notebook for better Colab UI
    val_loader_tqdm = tqdm(val_loader, desc="Validating", unit="batch", leave=False)

    with torch.no_grad():
        for ir_spectra_batch, smiles_batch in val_loader_tqdm:
            ir_spectra_tensor = ir_spectra_batch.to(device)

            # --- Logic identical to local script: augment and tokenize one-by-one ---
            current_smiles_batch = [augment_smiles(s, set(), sme) for s in smiles_batch]
            tokenizer = smiles_model.smiles_tokenizer
            encoded_smiles = [tokenizer.encode_plus(text=s, max_length=smiles_model.smiles_maxlen, padding='max_length',
                                                    truncation=True, return_tensors='pt') for s in current_smiles_batch]
            input_ids = torch.cat([item['input_ids'] for item in encoded_smiles], dim=0).to(device)
            attention_mask = torch.cat([item['attention_mask'] for item in encoded_smiles], dim=0).to(device)
            lengths = attention_mask.sum(dim=1)

            with torch.amp.autocast(device_type=device.type):
                smiles_features = smiles_model.encode((input_ids, attention_mask), lengths)
                ir_features = ir_model(ir_spectra_tensor)
                result_smiles_features.append(smiles_features)
                result_ir_features.append(ir_features)

                t = torch.exp(smiles_model.t_prime)
                b = smiles_model.bias
                logits = torch.matmul(ir_features, smiles_features.T) * t + b
                n = logits.size(0)
                labels = 2 * torch.eye(n, device=device) - torch.ones(n, n, device=device)
                loss = -torch.sum(F.logsigmoid(labels * logits)) / n

            running_loss += loss.item() * ir_spectra_tensor.size(0)
            val_loader_tqdm.set_postfix(loss=loss.item())

    # Calculate Top-1 accuracy
    all_smiles_features = torch.cat(result_smiles_features, 0)
    all_ir_features = torch.cat(result_ir_features, 0)
    logits_full = torch.matmul(all_ir_features, all_smiles_features.T)
    top1_indices = torch.argmax(logits_full, dim=1)
    correct_matches = (top1_indices == torch.arange(len(top1_indices), device=device)).sum().item()
    total_samples = len(top1_indices)
    top_1_ratio = correct_matches / total_samples if total_samples > 0 else 0
    val_loss = running_loss / len(val_loader.dataset)
    return val_loss, top_1_ratio


def train_model_colab(num_epochs, smiles_model, ir_model, train_loader, val_loader, optimizer, schedulers, scaler, device):
    best_models_tracker = []
    training_losses, validation_losses = [], []

    for epoch in range(num_epochs):
        smiles_model.train()
        ir_model.train()
        running_loss = 0.0
        # Using tqdm.notebook for better Colab UI
        train_loader_tqdm = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{num_epochs}", unit="batch")

        for ir_spectra_batch, smiles_batch in train_loader_tqdm:
            ir_spectra_tensor = ir_spectra_batch.to(device)

            # --- Logic identical to local script: augment and tokenize one-by-one ---
            current_smiles_batch = [augment_smiles(s, set(), sme) for s in smiles_batch]
            tokenizer = smiles_model.smiles_tokenizer
            encoded_smiles = [tokenizer.encode_plus(text=s, max_length=smiles_model.smiles_maxlen, padding='max_length',
                                                    truncation=True, return_tensors='pt') for s in current_smiles_batch]
            input_ids = torch.cat([item['input_ids'] for item in encoded_smiles], dim=0).to(device)
            attention_mask = torch.cat([item['attention_mask'] for item in encoded_smiles], dim=0).to(device)
            lengths = attention_mask.sum(dim=1)

            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=device.type):
                smiles_features = smiles_model.encode((input_ids, attention_mask), lengths)
                ir_features = ir_model(ir_spectra_tensor)
                t = torch.exp(smiles_model.t_prime)
                b = smiles_model.bias
                logits = torch.matmul(ir_features, smiles_features.T) * t + b
                n = logits.size(0)
                labels = 2 * torch.eye(n, device=device) - torch.ones(n, n, device=device)
                loss = -torch.sum(F.logsigmoid(labels * logits)) / n

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item()
            train_loader_tqdm.set_postfix(loss=loss.item())

        epoch_loss = running_loss / len(train_loader)
        training_losses.append(epoch_loss)
        val_loss, top_1_ratio = validate_model(smiles_model, ir_model, val_loader, device)
        validation_losses.append(val_loss)

        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch+1}/{num_epochs} -> Train Loss: {epoch_loss:.4f}, Val Loss: {val_loss:.4f}, Top-1 Ratio: {top_1_ratio:.4f}, LR: {current_lr:.2e}")

        # Update learning rate
        if epoch < 10: schedulers['warmup'].step()
        else: schedulers['cosine'].step(epoch - 10)

        # Save top 5 best models
        current_model_info = (top_1_ratio, epoch + 1, None, None)
        if len(best_models_tracker) < 5:
            best_models_tracker.append(current_model_info)
        elif top_1_ratio > min(best_models_tracker, key=lambda x: x[0])[0]:
            worst_model_info = min(best_models_tracker, key=lambda x: x[0])
            if worst_model_info[2] and os.path.exists(worst_model_info[2]): os.remove(worst_model_info[2])
            if worst_model_info[3] and os.path.exists(worst_model_info[3]): os.remove(worst_model_info[3])
            best_models_tracker.remove(worst_model_info)
            best_models_tracker.append(current_model_info)

        for i, (ratio, ep, _, _) in enumerate(best_models_tracker):
            if ep == epoch + 1:
                prefix = f"{dataset_type.lower()}"
                smiles_path = os.path.join(MODEL_OUTPUT_PATH, f'{prefix}_smiles_epoch_{ep}_ratio_{ratio:.4f}.pth')
                ir_path = os.path.join(MODEL_OUTPUT_PATH, f'{prefix}_ir_epoch_{ep}_ratio_{ratio:.4f}.pth')
                torch.save(smiles_model.state_dict(), smiles_path)
                torch.save(ir_model.state_dict(), ir_path)
                best_models_tracker[i] = (ratio, ep, smiles_path, ir_path)
                print(f"  -> 💾 Checkpoint saved for epoch {ep} with Top-1 Ratio: {ratio:.4f}")

    print("\n✅ Training complete.")
    print("🏆 Best validation models saved:")
    for ratio, epoch, _, _ in sorted(best_models_tracker, key=lambda x: x[0], reverse=True):
        print(f"  - Epoch {epoch}: Top-1 Ratio = {ratio:.4f}")

    return {'training_losses': training_losses, 'validation_losses': validation_losses}

print("✅ Core training and validation functions defined.")

print(f"SmilesModel trainable parameters: {count_parameters(smiles_model):,}")
print(f"IR_model trainable parameters: {count_parameters(ir_model):,}")
print("✅ Models and utilities initialized successfully.")
print("\n\n🎉🎉🎉 Full initialization is complete! 🎉🎉🎉")

### **Step 3: Choose Your Battlefield** 🎯

<div style="background-color: #e6f7ff; border-left: 6px solid #1890ff; padding: 15px; margin: 15px 0; border-radius: 5px;">
    <h4 style="color: #0050b3; margin-top: 0; font-size: 1.1em;">
        Interactive Data Selection
    </h4>
    <p>
        Now it's time to select your training data. This interactive cell is designed to be efficient by only loading the specific dataset you choose into memory.
    </p>
    <ul style="margin-top: 0; padding-left: 20px;">
        <li><strong>MD (Molecular Dynamics)</strong>: A dataset generated from molecular dynamics simulations.</li>
        <li><strong>DFT (Density Functional Theory)</strong>: A high-quality dataset generated from density functional theory calculations.</li>
    </ul>
    <p style="font-weight: bold;">
        Use the dropdown menu in the code cell below to make your selection. You can re-run the cell to switch datasets at any time.
    </p>
</div>

In [ ]:
# @title 3. Interactive Data Selection and Loading { run: "auto" }
# @markdown Please select the dataset you wish to train on. **Only the selected dataset will be loaded into memory.**
#@markdown *   **`batch_size`**: Our final model used a batch size of `208`. In Colab, the optimal value depends on your GPU memory. Training with batch size 208 on a RTX 4090 GPU (24G) takes approximately **30 minutes per epoch**. If you have to use a smaller `batch_size` due to memory constraints, the training time per epoch will be longer. If you encounter "Out of Memory" errors, reduce this value.

# --- User Selection Widget ---
dataset_type = "MD"  # @param ["MD", "DFT"]

# --- Dataset Path Configuration ---
DATA_PATHS = {
    "MD": {
        "base_path": os.path.join(PROJECT_ROOT, 'data', 'pretrain_data', 'colab_MD_data'),
        "prefix": "USTPO"
    },
    "DFT": {
        "base_path": os.path.join(PROJECT_ROOT, 'data', 'pretrain_data', 'Density functional simulation data'),
        "prefix": "QM9S_DFT"
    }
}

# --- Helper Function and Class Definitions ---
def load_data(smiles_path, ir_path):
    print(f"  > Loading SMILES: {os.path.basename(smiles_path)}")
    with open(smiles_path, 'r') as f:
        smiles = f.read().splitlines()
    print(f"  > Loading IR spectra: {os.path.basename(ir_path)}")
    ir = torch.load(ir_path, map_location='cpu')
    return smiles, ir

class IRSmilesDataset(Dataset):
    def __init__(self, ir_spectra, smiles):
        self.ir_spectra = ir_spectra
        self.smiles = smiles
    def __len__(self):
        return len(self.smiles)
    def __getitem__(self, idx):
        return self.ir_spectra[idx], self.smiles[idx]

# --- Conditional Data Loading ---
print(f"\n--- Preparing data for the [{dataset_type}] dataset ---")
config = DATA_PATHS[dataset_type]
base_path = config["base_path"]
prefix = config["prefix"]

train_smiles_path = os.path.join(base_path, f"{prefix}_train_smiles.txt")
train_ir_path = os.path.join(base_path, f"{prefix}_train_ir.pt")
val_smiles_path = os.path.join(base_path, f"{prefix}_val_smiles.txt")
val_ir_path = os.path.join(base_path, f"{prefix}_val_ir.pt")

smiles_train, ir_train = load_data(train_smiles_path, train_ir_path)
smiles_val, ir_val = load_data(val_smiles_path, val_ir_path)
print(f"✅ [{dataset_type}] dataset loaded. Training samples: {len(smiles_train)}.")

# --- Create Datasets and DataLoaders ---
train_dataset = IRSmilesDataset(ir_train, smiles_train)
val_dataset = IRSmilesDataset(ir_val, smiles_val)

batch_size = 1      # @param {type:"integer"}
# In Colab, set num_workers=2 is a good start. Set to 0 if you encounter issues.
num_workers = 0       # @param {type:"integer"}

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, num_workers=num_workers, pin_memory=True)
print(f"✅ DataLoaders created. Batch size: {batch_size}, Workers: {num_workers}")

### **Step 4: Ignite the Engines!** 🚀

<div style="background-color: #e6f7ff; border-left: 6px solid #1890ff; padding: 15px; margin: 15px 0; border-radius: 5px;">
<h4 style="color: #0050b3; margin-top: 0; font-size: 1.1em;">
Start the Training Process
</h4>
<p>
All systems are go! With your models initialized and data loaded, this is the final step where we kick off the actual training. Here's what will happen:
</p>
<ul style="margin-top: 0; padding-left: 20px;">
<li><strong>⚙️ Control Training Duration</strong>: You can adjust the <code>num_epochs</code> variable in the code to set how long the training runs.</li>
<li><strong>📊 Monitor Live Progress</strong>: The script will provide real-time updates, so you can watch your model learn.</li>
<li><strong>💾 Auto-Save Checkpoints</strong>: The best-performing models will be automatically saved to your Google Drive.</li>
</ul>
<p style="font-style: italic;">
<strong>Note:</strong> Model weights and loss curves will be saved to <code>/content/drive/MyDrive/colab_repos/CSU-IR/CSU-IR/check_points/colab_trained</code>.
</p>
<p style="font-weight: bold;">
When you're ready, run the cell below to begin training!
</p>
</div>

In [ ]:
# @title 4. Start Training!
# @markdown Run this cell to start the training process for the dataset you selected above.

# --- Training Hyperparameters ---
num_epochs = 80 # @param {type:"integer"}

print(f"--- Starting model training for {num_epochs} Epochs ---")
print(f"--- Training on: [{dataset_type}] Dataset ---")

# Clear CUDA cache before starting
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()

# --- Execute the Training Loop ---
history = train_model_colab(
    num_epochs=num_epochs,
    smiles_model=smiles_model,
    ir_model=ir_model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    schedulers={'warmup': scheduler_warmup, 'cosine': scheduler_cosine},
    scaler=scaler,
    device=device
)

# --- Save Final Results ---
loss_save_path = os.path.join(MODEL_OUTPUT_PATH, f'{dataset_type.lower()}_loss_history.json')
with open(loss_save_path, 'w') as f:
    json.dump(history, f)
print(f"\nTraining loss history saved to: {loss_save_path}")

print("\n🎉🎉🎉 Training finished successfully! 🎉🎉🎉")